# 01 File Integrity

P0 validation for endpoint presence, file size, zero-row files, and corruption.

In [1]:
from pathlib import Path
import sys
from datetime import datetime
import json

import pandas as pd
import plotly.express as px

ROOT = Path.cwd()
while not (ROOT / "configs" / "pipeline_config.yaml").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

SHARED = ROOT / "eda" / "shared" / "scripts"
if str(SHARED) not in sys.path:
    sys.path.insert(0, str(SHARED))

from config import (
    RAW_DATA_PATH,
    EXPECTED_ENDPOINTS,
    TELEMETRY_ENDPOINTS,
)
from file_utils import build_file_inventory, iter_csv_endpoint

NOTEBOOK_NAME = "01_file_integrity"
OUTPUT_TABLES = ROOT / "eda" / "bronze" / "outputs" / "tables" / NOTEBOOK_NAME
OUTPUT_CHARTS = ROOT / "eda" / "bronze" / "outputs" / "charts" / NOTEBOOK_NAME
OUTPUT_REPORTS = ROOT / "eda" / "bronze" / "outputs" / "reports" / NOTEBOOK_NAME
INSIGHTS = ROOT / "eda" / "bronze" / "insights"
CHECKPOINTS = ROOT / "eda" / "bronze" / "checkpoints"
for path in [OUTPUT_TABLES, OUTPUT_CHARTS, OUTPUT_REPORTS, INSIGHTS, CHECKPOINTS]:
    path.mkdir(parents=True, exist_ok=True)

def write_report(name: str, payload: dict) -> None:
    (OUTPUT_REPORTS / f"{name}.json").write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")

def write_insight(filename: str, title: str, summary: str, observations: list[str], issues: list[str], recommendations: list[str], next_steps: list[str]) -> None:
    content = f"# {title}\n\n"
    content += f"**Generated at:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    content += f"## Summary\n\n{summary}\n\n"
    content += "## Key Observations\n\n" + "\n".join(f"- {item}" for item in observations) + "\n\n"
    content += "## Issues\n\n" + ("\n".join(f"- {item}" for item in issues) if issues else "- None") + "\n\n"
    content += "## Recommendations\n\n" + "\n".join(f"- {item}" for item in recommendations) + "\n\n"
    content += "## Next Steps\n\n" + "\n".join(f"- {item}" for item in next_steps) + "\n"
    (INSIGHTS / filename).write_text(content, encoding="utf-8")

def p0_status_from_severity(df: pd.DataFrame) -> str:
    if df.empty or "severity" not in df.columns:
        return "FAIL"
    blockers = df[df["severity"].eq("BLOCKER")]
    return "FAIL" if not blockers.empty else "PASS"

print("=" * 72)
print(f"BRONZE VALIDATION - {NOTEBOOK_NAME}")
print(f"Start time: {datetime.now()}")
print(f"Raw data path: {RAW_DATA_PATH}")
print("=" * 72)

BRONZE VALIDATION - 01_file_integrity
Start time: 2026-06-01 18:19:36.922970
Raw data path: D:\F1_WinRate_Predictor\data\raw


In [2]:
inventory = build_file_inventory(RAW_DATA_PATH, EXPECTED_ENDPOINTS)

inventory["endpoint_group"] = inventory["endpoint"].map(
    lambda endpoint: "Telemetry" if endpoint in TELEMETRY_ENDPOINTS else "Operational"
)

inventory["severity"] = inventory.apply(
    lambda row: "BLOCKER" if row["error"] == "endpoint_missing" or row["corrupt"] or row["zero_rows"] else "INFO",
    axis=1,
)

inventory["status"] = inventory["severity"].map(lambda s: "FAIL" if s == "BLOCKER" else "PASS")

inventory.to_csv(OUTPUT_TABLES / "file_integrity.csv", index=False)

summary = (
    inventory.groupby(["endpoint_group", "endpoint"], dropna=False)
    .agg(
        files=("relative_path", lambda s: int((s != "").sum())),
        rows=("row_count", "sum"),
        blockers=("severity", lambda s: int((s == "BLOCKER").sum()))
    )
    .reset_index()
)

summary["status"] = summary["blockers"].map(lambda b: "FAIL" if b > 0 else "PASS")
summary.to_csv(OUTPUT_TABLES / "file_integrity_summary.csv", index=False)

display(summary)

,endpoint_group,endpoint,files,rows,blockers,status
0,Operational,drivers,140,2837,0,PASS
1,Operational,intervals,68,1393827,0,PASS
2,Operational,laps,135,84072,0,PASS
3,Operational,meetings,3,76,0,PASS
4,Operational,overtakes,68,13995,0,PASS
5,Operational,pit,132,7871,0,PASS
6,Operational,position,136,115021,0,PASS
7,Operational,race_control,136,9736,0,PASS
8,Operational,session_result,136,2740,0,PASS
9,Operational,sessions,140,140,0,PASS


In [3]:
fig = px.bar(
    summary,
    x="endpoint",
    y="files",
    color="endpoint_group",
    title="Bronze File Integrity: File Coverage by Endpoint Group",
    labels={"files": "Files", "endpoint": "Endpoint"},
)
fig.update_xaxes(tickangle=35)
fig.write_html(OUTPUT_CHARTS / "row_coverage.html", include_plotlyjs="cdn")
try:
    fig.write_image(OUTPUT_CHARTS / "row_coverage.png")
except Exception:
    pass
fig.show()

In [4]:
failed = inventory[inventory["status"] == "FAIL"]
report = {
    "notebook": NOTEBOOK_NAME,
    "timestamp": datetime.now().isoformat(),
    "p0_status": p0_status_from_severity(inventory),
    "files_scanned": int(len(inventory)),
    "failed_records": failed.to_dict("records"),
}
write_report("file_integrity", report)
write_insight(
    "01_file_integrity_insights.md",
    "File Integrity Insights",
    f"Scanned {len(inventory)} raw files across {inventory['endpoint'].nunique()} endpoints.",
    [
        f"Corrupt files: {int(inventory['corrupt'].sum())}",
        f"Zero-row files: {int(inventory['zero_rows'].sum())}",
        f"Telemetry rows: {int(inventory[inventory['endpoint_group'].eq('Telemetry')]['row_count'].sum()):,}",
    ],
    [f"{row.endpoint}: {row.error or 'failed integrity'}" for row in failed.itertuples()],
    ["Keep telemetry validation streaming-only.", "Proceed to schema validation if P0 status is PASS."],
    ["Run 02_schema_validation.ipynb"],
)
print(report)

{'notebook': '01_file_integrity', 'timestamp': '2026-06-01T18:19:41.988738', 'p0_status': 'PASS', 'files_scanned': 1653, 'failed_records': []}
